# Práctica 4 — MongoDB y consultas NoSQL

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Crear una base documental en MongoDB Atlas y ejecutar CRUD, filtros, agregaciones e índices.


In [ ]:
!pip -q install pymongo[srv]
import os
from getpass import getpass
from pymongo import MongoClient, ASCENDING

uri = os.environ.get("MONGO_URI") or getpass("MONGO_URI de Atlas: " )
client = MongoClient(uri, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
db = client["itla_bigdata"]
ventas = db["ventas"]
ventas.delete_many({"curso": "2026-C-3"})
print("Conexión verificada")


In [ ]:
documentos = [
 {"cliente":"Ana", "ciudad":"Santo Domingo", "producto":"Laptop", "monto":45000, "curso":"2026-C-3"},
 {"cliente":"Luis", "ciudad":"Santiago", "producto":"Mouse", "monto":800, "curso":"2026-C-3"},
 {"cliente":"Marta", "ciudad":"Santo Domingo", "producto":"Teclado", "monto":1500, "curso":"2026-C-3"},
 {"cliente":"Ana", "ciudad":"Santo Domingo", "producto":"Mouse", "monto":800, "curso":"2026-C-3"},
]
ventas.insert_many(documentos)
ventas.create_index([("ciudad", ASCENDING), ("monto", ASCENDING)])
list(ventas.find({"curso":"2026-C-3"}, {"_id":0}))


In [ ]:
# READ y filtros
print("Monto > 1000:", list(ventas.find({"monto":{"$gt":1000},"curso":"2026-C-3"},{"_id":0})))
print("Santo Domingo:", list(ventas.find({"ciudad":"Santo Domingo","curso":"2026-C-3"},{"_id":0})))

# UPDATE y DELETE
ventas.update_one({"cliente":"Luis","curso":"2026-C-3"},{"$set":{"monto":850}})
ventas.delete_one({"cliente":"Marta","curso":"2026-C-3"})


In [ ]:
# Agregación por ciudad
pipeline = [
 {"$match":{"curso":"2026-C-3"}},
 {"$group":{"_id":"$ciudad","ventas":{"$sum":1},"monto_total":{"$sum":"$monto"}}},
 {"$sort":{"monto_total":-1}}
]
list(ventas.aggregate(pipeline))
